# LeetCode 2044 — Count Number of Maximum Bitwise-OR Subsets bằng Python

Link đề: https://leetcode.com/problems/count-number-of-maximum-bitwise-or-subsets/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking, Bit Manipulation, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def countMaxOrSubsets(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra **OR lớn nhất** của một tập con chính là OR của **toàn bộ mảng** (OR chỉ bật thêm bit, không bao giờ tắt).
2. Liệt kê `2ⁿ` tập con bằng **bitmask** và bằng **đệ quy chọn/không chọn**; hiểu vì sao `n ≤ 16` cho phép điều đó.
3. Tối ưu liệt kê bằng **DP trên mask** với `lowbit` (`O(2ⁿ)` thay vì `O(n·2ⁿ)`), và bằng **DP đếm theo giá trị OR** (`O(n · số giá trị OR khác nhau)`).
4. Cắt tỉa: khi OR hiện tại đã đạt mục tiêu, mọi cách chọn phần còn lại đều đạt ⇒ cộng `2^(còn lại)` một lần.
5. Phân biệt tập con theo **chỉ số** (đề yêu cầu) với tập con theo **giá trị**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, tìm **OR theo bit lớn nhất** có thể của một tập con và trả về **số tập con khác rỗng khác nhau** có OR bằng giá trị lớn nhất đó.

Mảng `a` là tập con của `b` nếu thu được `a` bằng cách xóa một số (có thể 0) phần tử của `b`. Hai tập con **khác nhau** nếu tập **chỉ số** được chọn khác nhau. OR của mảng `a` là `a[0] | a[1] | … | a[len−1]`.

**Ví dụ 1:** `[3,1]` → `2` (OR lớn nhất 3: `[3]`, `[3,1]`).

**Ví dụ 2:** `[2,2,2]` → `7` (mọi `2³ − 1` tập con khác rỗng đều có OR 2).

**Ví dụ 3:** `[3,2,1,5]` → `6` (OR lớn nhất 7: `[3,5]`, `[3,1,5]`, `[3,2,5]`, `[3,2,1,5]`, `[2,5]`, `[2,1,5]`).

**Ràng buộc:** `1 ≤ n ≤ 16`; `1 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from functools import reduce, lru_cache
from itertools import combinations
from operator import or_
from typing import List

examples = [([3, 1], 2), ([2, 2, 2], 7), ([3, 2, 1, 5], 6)]
for nums, exp in examples:
    print(f"nums={nums} -> kỳ vọng {exp}")

nums=[3, 1] -> kỳ vọng 2
nums=[2, 2, 2] -> kỳ vọng 7
nums=[3, 2, 1, 5] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1 Phép OR theo bit

`a | b` bật bit `k` nếu `a` **hoặc** `b` có bit `k`. Tính chất: giao hoán, kết hợp, `a | a = a`, và `a | b ≥ max(a, b)` — OR **không bao giờ giảm** khi thêm phần tử.

In [2]:
for a, b in [(3, 5), (2, 2), (1, 4)]:
    print(f"{a} = {a:04b}, {b} = {b:04b} -> {a} | {b} = {a | b} = {a | b:04b}")

3 = 0011, 5 = 0101 -> 3 | 5 = 7 = 0111
2 = 0010, 2 = 0010 -> 2 | 2 = 2 = 0010
1 = 0001, 4 = 0100 -> 1 | 4 = 5 = 0101


### 3.2 Bitmask biểu diễn tập con

Với `n` phần tử, mỗi số `mask ∈ [0, 2ⁿ)` là một tập con: bit `i` bật ⇔ chọn `nums[i]`. `mask = 0` là tập rỗng.

### 3.3 `lowbit` và chỉ số của nó

`mask & −mask` là bit thấp nhất đang bật; `(mask & −mask).bit_length() − 1` là chỉ số của nó. Bỏ bit thấp nhất: `mask & (mask − 1)`.

In [3]:
mask = 0b10110
low = mask & -mask
print(f"mask={mask:05b}, lowbit={low:05b} (chỉ số {low.bit_length() - 1}), bỏ lowbit -> {mask & (mask - 1):05b}")

mask=10110, lowbit=00010 (chỉ số 1), bỏ lowbit -> 10100


### 3.4 Số tập con

`n` phần tử có `2ⁿ` tập con (kể cả rỗng). Với `n = 16`: `65 536` — đủ nhỏ để liệt kê hết, kể cả `O(n · 2ⁿ) ≈ 10⁶`.

### 3.5 "Tập con bit" và liệt kê các mask con

`a` là **tập con bit** của `t` nếu mọi bit bật của `a` cũng bật trong `t`, tức `a | t == t` (tương đương `a & t == a`). Mọi mask con của `t` có thể liệt kê theo thứ tự giảm dần bằng mẹo `sub = (sub − 1) & t` — dùng ở Bài tập 5. Số mask con của `t` là `2^popcount(t)`.

In [4]:
t = 0b1011
subs, sub = [], t
while True:
    subs.append(sub)
    if sub == 0:
        break
    sub = (sub - 1) & t
print(f"các mask con của {t:04b}:", [f"{x:04b}" for x in subs], "| số lượng:", len(subs), "= 2^", bin(t).count("1"))
print("5 là tập con bit của 7?", 5 | 7 == 7, "| 5 của 6?", 5 | 6 == 6)

các mask con của 1011: ['1011', '1010', '1001', '1000', '0011', '0010', '0001', '0000'] | số lượng: 8 = 2^ 3
5 là tập con bit của 7? True | 5 của 6? False


## 4. Ý tưởng nền tảng / trực giác

**Bước 1 — OR lớn nhất là gì?** Vì OR không giảm khi thêm phần tử, tập con có OR lớn nhất là **cả mảng**: `target = nums[0] | … | nums[n−1]`. Mọi tập con khác có OR `≤ target` (là "tập con bit" của `target`).

**Bước 2 — đếm.** Đếm số tập con khác rỗng có OR đúng bằng `target`. Với `n ≤ 16`, liệt kê hết.

Tính tay **Ví dụ 3** `[3,2,1,5]` (`target = 3|2|1|5 = 7`):

| tập con | OR | = 7? |
|---|---|---|
| `[5]` | 5 | ✗ |
| `[3,5]` | 7 | ✓ |
| `[2,5]` | 7 | ✓ |
| `[1,5]` | 5 | ✗ |
| `[3,2,1]` | 3 | ✗ |
| `[3,1,5]`, `[3,2,5]`, `[2,1,5]`, `[3,2,1,5]` | 7 | ✓ ×4 |

Tổng cộng `6`. Để ý: phải có `5` (bit 2) **và** một trong `3`, `2` (bit 1).

In [5]:
nums = [3, 2, 1, 5]
target = reduce(or_, nums)
hits = [[nums[i] for i in range(4) if m >> i & 1] for m in range(1, 16) if reduce(or_, (nums[i] for i in range(4) if m >> i & 1)) == target]
print("target =", target, "| các tập con đạt:", hits, "| số lượng:", len(hits))

target = 7 | các tập con đạt: [[3, 5], [2, 5], [3, 2, 5], [3, 1, 5], [2, 1, 5], [3, 2, 1, 5]] | số lượng: 6


## 5. Cách 1 — Brute Force / baseline: duyệt mọi mask, tính OR từng mask

Với mỗi `mask ∈ [1, 2ⁿ)`, OR các phần tử được chọn trong `O(n)`.

In [6]:
def count_bruteforce(nums):
    n = len(nums); target = reduce(or_, nums); cnt = 0
    for mask in range(1, 1 << n):
        v = 0
        for i in range(n):
            if mask >> i & 1:
                v |= nums[i]
        cnt += v == target
    return cnt

for nums, exp in examples:
    got = count_bruteforce(nums); print(got, "kỳ vọng", exp); assert got == exp

2 kỳ vọng 2
7 kỳ vọng 7
6 kỳ vọng 6


**Độ phức tạp:** `O(n · 2ⁿ)` = `16 · 65 536 ≈ 10⁶` phép — chấp nhận được với `n ≤ 16`, nhưng trong Python mất vài trăm ms. Không gian `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (target):** `target = OR toàn mảng` — không cần thử để tìm "OR lớn nhất".

**Insight 2 (DP trên mask):** `OR(mask) = OR(mask bỏ lowbit) | nums[chỉ số lowbit]` ⇒ tính OR của mọi mask trong `O(1)` mỗi mask ⇒ `O(2ⁿ)`.

**Insight 3 (DP theo giá trị):** duyệt từng phần tử, giữ `dp[v]` = số tập con (đã xét) có OR bằng `v`. Thêm `x`: mỗi tập con cũ có thể không chọn `x` (giữ `v`) hoặc chọn (thành `v | x`). Số giá trị `v` khác nhau thường nhỏ hơn nhiều so với `2ⁿ`.

**Insight 4 (cắt tỉa DFS):** đang xét phần tử `i` mà OR hiện tại đã bằng `target` ⇒ mọi cách chọn `n − i` phần tử còn lại đều giữ OR = target ⇒ cộng `2^(n−i)` và dừng nhánh.

**Invariant (DP theo giá trị):** sau khi xét `k` phần tử đầu, `dp[v]` = số tập con (kể cả rỗng) của `nums[:k]` có OR `v`.

### 6.1 Minh họa: số giá trị OR khác nhau nhỏ hơn nhiều so với số tập con

Với `nums` ngẫu nhiên `n = 16`, đếm số khóa của `dp` sau mỗi bước. Mỗi giá trị OR là "tập con bit" của `target` nên số khóa `≤ 2^(số bit của target)` và thường chỉ vài trăm, so với `65 536` tập con.

In [7]:
rng = random.Random(61)
nums = [rng.randint(1, 10 ** 5) for _ in range(16)]
dp = {0: 1}; sizes = []
for x in nums:
    nd = dict(dp)
    for v, c in dp.items():
        nd[v | x] = nd.get(v | x, 0) + c
    dp = nd; sizes.append(len(dp))
print("số khóa dp sau mỗi phần tử:", sizes)
print("target có", bin(reduce(or_, nums)).count("1"), "bit bật; tổng số tập con:", 2 ** 16)

số khóa dp sau mỗi phần tử: [2, 4, 7, 12, 17, 26, 40, 53, 78, 92, 116, 120, 154, 171, 197, 227]
target có 17 bit bật; tổng số tập con: 65536


## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS chọn/không chọn (có cắt tỉa)

In [8]:
def count_dfs(nums):
    n = len(nums); target = reduce(or_, nums)
    def dfs(i, cur):
        if cur == target:
            return 1 << (n - i)              # mọi cách chọn phần còn lại đều đạt
        if i == n:
            return 0
        return dfs(i + 1, cur | nums[i]) + dfs(i + 1, cur)
    return dfs(0, 0)

print([count_dfs(nums) for nums, _ in examples])

[2, 7, 6]


`dfs(i, cur)` = số cách chọn tập con của `nums[i:]` sao cho `cur | OR(phần chọn) = target`. Khi `cur = target` có `2^(n−i)` cách (kể cả không chọn gì thêm). Vì `target ≥ 1`, tập rỗng ở gốc (`cur = 0`) không bao giờ được đếm.

**Độ phức tạp Cách 2:** tệ nhất `O(2ⁿ)` lời gọi, thường ít hơn nhờ cắt tỉa. Độ sâu đệ quy `n ≤ 16`.

### 7.2 Cách 3 — DP trên mask với `lowbit`

In [9]:
def count_mask_dp(nums):
    n = len(nums); target = reduce(or_, nums)
    orv = [0] * (1 << n); cnt = 0
    for mask in range(1, 1 << n):
        low = mask & -mask
        orv[mask] = orv[mask ^ low] | nums[low.bit_length() - 1]
        cnt += orv[mask] == target
    return cnt

print([count_mask_dp(nums) for nums, _ in examples])

[2, 7, 6]


**Độ phức tạp Cách 3:** `O(2ⁿ)` thời gian, `O(2ⁿ)` bộ nhớ (`65 536` số).

### 7.3 Cách 4 — DP đếm theo giá trị OR (bản nộp)

In [10]:
class Solution:
    def countMaxOrSubsets(self, nums: List[int]) -> int:
        dp = {0: 1}                          # OR -> số tập con (kể cả rỗng)
        for x in nums:
            nd = dict(dp)                    # không chọn x
            for v, c in dp.items():
                nd[v | x] = nd.get(v | x, 0) + c   # chọn x
            dp = nd
        target = 0
        for x in nums:
            target |= x
        return dp[target]

print([Solution().countMaxOrSubsets(nums) for nums, _ in examples])

[2, 7, 6]


Vì `target ≥ 1`, khóa `target ≠ 0` nên `dp[target]` không bao gồm tập rỗng.

**Độ phức tạp Cách 4:** `O(n · K)` với `K` = số giá trị OR khác nhau (`≤ min(2ⁿ, 2^popcount(target))`). Bộ nhớ `O(K)`.

### 7.4 Cách 5 — `itertools.combinations` theo vị trí

`combinations(nums, r)` sinh các bộ theo **vị trí** (phần tử trùng giá trị ở vị trí khác nhau vẫn là bộ khác nhau) — khớp định nghĩa "khác chỉ số" của đề. Ngắn gọn nhưng vẫn `O(n · 2ⁿ)`.

In [11]:
def count_combinations(nums):
    t = reduce(or_, nums)
    return sum(1 for r in range(1, len(nums) + 1) for c in combinations(nums, r) if reduce(or_, c) == t)

print([count_combinations(nums) for nums, _ in examples], "| [2,2,2] sinh", len(list(combinations([2, 2, 2], 2))), "bộ cỡ 2")

[2, 7, 6] | [2,2,2] sinh 3 bộ cỡ 2


## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 3** `[3,2,1,5]`:

In [12]:
def trace_dp(nums):
    dp = {0: 1}
    print("ban đầu:", dp)
    for x in nums:
        nd = dict(dp)
        for v, c in dp.items():
            nd[v | x] = nd.get(v | x, 0) + c
        dp = nd
        print(f"thêm {x}: {dict(sorted(dp.items()))}")
    t = reduce(or_, nums); print(f"target = {t} -> dp[target] = {dp[t]}")
    return dp[t]

assert trace_dp([3, 2, 1, 5]) == 6

ban đầu: {0: 1}
thêm 3: {0: 1, 3: 1}
thêm 2: {0: 1, 2: 1, 3: 2}
thêm 1: {0: 1, 1: 1, 2: 1, 3: 5}
thêm 5: {0: 1, 1: 1, 2: 1, 3: 5, 5: 2, 7: 6}
target = 7 -> dp[target] = 6


| bước | thêm | dp (OR: số tập con) |
|---|---|---|
| 0 | — | `{0:1}` |
| 1 | 3 | `{0:1, 3:1}` |
| 2 | 2 | `{0:1, 2:1, 3:2}` (`3|2 = 3`) |
| 3 | 1 | `{0:1, 1:1, 2:1, 3:5}` (`2|1 = 3`, `3|1 = 3`) |
| 4 | 5 | `{0:1, 1:1, 2:1, 3:5, 5:2, 7:6}` |

Khi thêm `5`: `0|5 = 5`, `1|5 = 5` (2 tập), `2|5 = 7`, `3|5 = 7` (`1 + 5 = 6` tập) ⇒ `dp[7] = 6` ✓.

Trace thêm Cách 2 (DFS cắt tỉa) — in các lần cắt:

In [13]:
def trace_dfs(nums):
    n = len(nums); target = reduce(or_, nums); calls = [0]
    def dfs(i, cur, path):
        calls[0] += 1
        if cur == target:
            print(f"  cắt tại i={i}, đã chọn {path}: +2^{n - i} = {1 << (n - i)}"); return 1 << (n - i)
        if i == n:
            return 0
        return dfs(i + 1, cur | nums[i], path + [nums[i]]) + dfs(i + 1, cur, path)
    r = dfs(0, 0, []); print(f"tổng {r}, số lời gọi {calls[0]} (không cắt tỉa: {2 ** (n + 1) - 1})"); return r

assert trace_dfs([3, 2, 1, 5]) == 6

  cắt tại i=4, đã chọn [3, 2, 1, 5]: +2^0 = 1
  cắt tại i=4, đã chọn [3, 2, 5]: +2^0 = 1
  cắt tại i=4, đã chọn [3, 1, 5]: +2^0 = 1
  cắt tại i=4, đã chọn [3, 5]: +2^0 = 1
  cắt tại i=4, đã chọn [2, 1, 5]: +2^0 = 1
  cắt tại i=4, đã chọn [2, 5]: +2^0 = 1
tổng 6, số lời gọi 31 (không cắt tỉa: 31)


Trace Cách 3 (DP trên mask) cho **Ví dụ 1** `[3,1]` và **Ví dụ 3**: mỗi mask lấy OR của mask "bỏ lowbit" rồi OR thêm đúng một phần tử. In các mask đạt target.

In [14]:
def trace_mask(nums):
    n = len(nums); t = reduce(or_, nums); orv = [0] * (1 << n); hits = 0
    for mask in range(1, 1 << n):
        low = mask & -mask; i = low.bit_length() - 1
        orv[mask] = orv[mask ^ low] | nums[i]
        hits += orv[mask] == t
        if n <= 2 or orv[mask] == t:
            print(f"mask={mask:0{n}b}: OR(mask bỏ bit {i}) = {orv[mask ^ low]} | nums[{i}]={nums[i]} -> {orv[mask]}{'  ✓' if orv[mask] == t else ''}")
    print("=> số mask đạt:", hits); return hits

assert trace_mask([3, 1]) == 2
assert trace_mask([3, 2, 1, 5]) == 6

mask=01: OR(mask bỏ bit 0) = 0 | nums[0]=3 -> 3  ✓
mask=10: OR(mask bỏ bit 1) = 0 | nums[1]=1 -> 1
mask=11: OR(mask bỏ bit 0) = 1 | nums[0]=3 -> 3  ✓
=> số mask đạt: 2
mask=1001: OR(mask bỏ bit 0) = 5 | nums[0]=3 -> 7  ✓
mask=1010: OR(mask bỏ bit 1) = 5 | nums[1]=2 -> 7  ✓
mask=1011: OR(mask bỏ bit 0) = 7 | nums[0]=3 -> 7  ✓
mask=1101: OR(mask bỏ bit 0) = 5 | nums[0]=3 -> 7  ✓
mask=1110: OR(mask bỏ bit 1) = 5 | nums[1]=2 -> 7  ✓
mask=1111: OR(mask bỏ bit 0) = 7 | nums[0]=3 -> 7  ✓
=> số mask đạt: 6


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — DP đếm theo giá trị OR.

In [15]:
class Solution:
    def countMaxOrSubsets(self, nums: List[int]) -> int:
        dp = {0: 1}
        for x in nums:
            nd = dict(dp)
            for v, c in dp.items():
                nd[v | x] = nd.get(v | x, 0) + c
            dp = nd
        target = 0
        for x in nums:
            target |= x
        return dp[target]

for nums, exp in examples:
    got = Solution().countMaxOrSubsets(nums); print(got, exp); assert got == exp

2 2
7 7
6 6


## 10. Vì sao thuật toán đúng

**OR lớn nhất = `target`.** Với mọi tập con `S`, `OR(S)` chỉ có các bit thuộc `target` ⇒ `OR(S) ≤ target` (về giá trị, vì là tập con bit). Tập con "cả mảng" đạt đúng `target`. Vậy OR lớn nhất là `target`.

**Cách 4 (DP).** Quy nạp theo số phần tử đã xét `k`. `k = 0`: chỉ có tập rỗng, OR `0` ⇒ `dp = {0: 1}`. Bước `k → k + 1` với phần tử `x`: mỗi tập con của `nums[:k+1]` hoặc không chứa `x` (là tập con của `nums[:k]`, OR giữ nguyên `v`), hoặc chứa `x` (tập con của `nums[:k]` hợp thêm `x`, OR thành `v | x`). Hai loại rời nhau và phủ hết ⇒ `nd[w] = dp[w] + Σ_{v : v|x = w} dp[v]` — đúng như code. Kết thúc, `dp[target]` đếm mọi tập con (theo **chỉ số**, vì mỗi phần tử được xét riêng) có OR `target`; tập rỗng có OR `0 ≠ target` nên không bị đếm. ∎

**Cách 2 (cắt tỉa).** Nếu `cur = target` ở bước `i`, mọi tập con `T ⊆ {i, …, n−1}` cho `cur | OR(T) = target` (OR không vượt `target` và không giảm) ⇒ có đúng `2^(n−i)` cách. Ngược lại tách theo chọn/không chọn `nums[i]`.

### 10.1 Kiểm chứng thực nghiệm

(a) Với mọi tập con, `OR(S)` là "tập con bit" của `target` (`OR(S) | target == target`); (b) tổng mọi `dp[v]` bằng `2ⁿ`; (c) bốn cách cho cùng đáp án.

In [16]:
rng = random.Random(1010)
for _ in range(300):
    n = rng.randint(1, 10); nums = [rng.randint(1, 40) for _ in range(n)]
    t = reduce(or_, nums)
    for mask in range(1, 1 << n):
        v = reduce(or_, (nums[i] for i in range(n) if mask >> i & 1))
        assert v | t == t and v <= t
    dp = {0: 1}
    for x in nums:
        nd = dict(dp)
        for v, c in dp.items(): nd[v | x] = nd.get(v | x, 0) + c
        dp = nd
    assert sum(dp.values()) == 2 ** n
    assert len({count_bruteforce(nums), count_dfs(nums), count_mask_dp(nums), Solution().countMaxOrSubsets(nums)}) == 1
print("(a) OR(S) ⊆ target, (b) Σ dp = 2^n, (c) 4 cách trùng nhau — 300 mảng")

(a) OR(S) ⊆ target, (b) Σ dp = 2^n, (c) 4 cách trùng nhau — 300 mảng


### 10.2 Một nhận xét thú vị: `dp[target]` luôn là số lớn nhất trong `dp`

**Mệnh đề.** Với mọi giá trị `v`, số tập con có OR `v` không vượt số tập con có OR `target`.

**Chứng minh (đơn ánh).** Cố định `v ≠ target`. Gọi `M` = tập chỉ số `j` mà `nums[j]` **không** là tập con bit của `v`. Mọi tập con `S` có `OR(S) = v` chỉ chứa phần tử là tập con bit của `v`, nên `S ∩ M = ∅`. Ánh xạ `S ↦ S ∪ M`: OR của ảnh là `v | OR(M)`; mọi bit của `target` nằm ngoài `v` đều thuộc một phần tử nào đó, và phần tử ấy nằm trong `M` ⇒ OR của ảnh là `target`. Ánh xạ đơn ánh vì `S = (S ∪ M) \ M`. ∎

Hệ quả: lỗi "trả `max(dp.values())`" tình cờ vẫn đúng với bài này — nhưng chỉ nhờ một mệnh đề không hiển nhiên; đừng dựa vào may mắn. Kiểm chứng thực nghiệm:

In [17]:
rng = random.Random(102)
for _ in range(3000):
    nums = [rng.randint(1, 31) for _ in range(rng.randint(1, 9))]
    dp = {0: 1}
    for x in nums:
        nd = dict(dp)
        for v, c in dp.items(): nd[v | x] = nd.get(v | x, 0) + c
        dp = nd
    t = reduce(or_, nums)
    assert dp[t] == max(dp.values())
    for v in dp:
        if v != t:
            M = [j for j, x in enumerate(nums) if x | v != v]
            assert reduce(or_, (nums[j] for j in M), v) == t
print("dp[target] = max(dp) và ánh xạ S -> S ∪ M cho OR = target: đúng trên 3000 mảng")

dp[target] = max(dp) và ánh xạ S -> S ∪ M cho OR = target: đúng trên 3000 mảng


## 11. Phân tích Time / Space Complexity

Gọi `n ≤ 16`, `K` = số giá trị OR khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mask, OR từng mask | `O(n · 2ⁿ)` | `O(1)` |
| 2. DFS cắt tỉa | `O(2ⁿ)` tệ nhất | `O(n)` (đệ quy) |
| 3. DP mask + `lowbit` | `O(2ⁿ)` | `O(2ⁿ)` |
| 4. DP theo giá trị | `O(n · K)` | `O(K)` |

In [18]:
rng = random.Random(2044)
cases = {"ngẫu nhiên": [rng.randint(1, 10 ** 5) for _ in range(16)],
         "lũy thừa 2": [1 << i for i in range(16)],
         "toàn bằng nhau": [7] * 16}
for label, nums in cases.items():
    row = []
    for name, f in [("brute", count_bruteforce), ("dfs", count_dfs), ("mask", count_mask_dp), ("dp", Solution().countMaxOrSubsets)]:
        t = time.perf_counter(); r = f(nums); row.append(f"{name} {(time.perf_counter() - t) * 1000:5.0f} ms")
    print(f"{label:<15} đáp án {r:>6} | " + " | ".join(row))

ngẫu nhiên      đáp án  59843 | brute    44 ms | dfs     1 ms | mask     7 ms | dp     0 ms


lũy thừa 2      đáp án      1 | brute    42 ms | dfs     7 ms | mask     6 ms | dp     8 ms


toàn bằng nhau  đáp án  65535 | brute    38 ms | dfs     0 ms | mask     6 ms | dp     0 ms


Với `"lũy thừa 2"`, mọi tập con có OR khác nhau ⇒ `K = 2ⁿ` — DP theo giá trị mất lợi thế. Với `"toàn bằng nhau"`, DFS cắt ngay sau phần tử đầu tiên.

## 12. Bộ test

In [19]:
ALL = {"brute": count_bruteforce, "dfs": count_dfs, "mask": count_mask_dp, "submit": Solution().countMaxOrSubsets}
def run_all(nums):
    res = {k: f(list(nums)) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (nums, res)
    return res["submit"]

for nums, exp in examples:
    assert run_all(nums) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [([1], 1), ([5, 5], 3), ([1, 2, 4], 1), ([1, 2, 3], 5), ([7, 1, 2, 4], 9), ([1] * 16, 2 ** 16 - 1),
        ([10 ** 5], 1), ([6, 3], 1)]
for nums, exp in edge:
    assert run_all(nums) == exp, (nums, run_all(nums))
rng = random.Random(20440)
for _ in range(300):
    run_all([rng.randint(1, rng.choice([3, 15, 10 ** 5])) for _ in range(rng.randint(1, 12))])
for _ in range(5):
    run_all([rng.randint(1, 10 ** 5) for _ in range(16)])
print("Ví dụ, 8 edge case, 300 mảng nhỏ, 5 mảng n = 16: OK")

Ví dụ, 8 edge case, 300 mảng nhỏ, 5 mảng n = 16: OK


### 12.1 Kiểm tra tính chất

- Hoán vị `nums` không đổi đáp án.
- Thêm một phần tử bằng `target`: mọi tập con chứa phần tử mới đều đạt (có `2ⁿ` tập như vậy), các tập không chứa nó giữ nguyên ⇒ đáp án mới `= cũ + 2ⁿ`.
- Thêm một bản sao của một phần tử có sẵn không làm đáp án giảm.

In [20]:
rng = random.Random(121)
f = Solution().countMaxOrSubsets
for _ in range(400):
    n = rng.randint(1, 10); nums = [rng.randint(1, 60) for _ in range(n)]
    a = f(nums); sh = nums[:]; rng.shuffle(sh)
    assert f(sh) == a
    t = reduce(or_, nums)
    assert f(nums + [t]) == a + 2 ** n
    assert f(nums + [rng.choice(nums)]) >= a
print("3 tính chất đúng trên 400 mảng")

3 tính chất đúng trên 400 mảng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm tập con theo **giá trị** (loại trùng)

In [21]:
def wrong_by_value(nums):
    vals = sorted(set(nums)); t = reduce(or_, vals)
    return sum(1 for r in range(1, len(vals) + 1) for c in combinations(vals, r) if reduce(or_, c) == t)   # SAI

for nums, exp in examples:
    print(f"{nums}: sai {wrong_by_value(nums)} | đúng {exp}")

[3, 1]: sai 2 | đúng 2
[2, 2, 2]: sai 1 | đúng 7
[3, 2, 1, 5]: sai 6 | đúng 6


**Vì sao sai:** đề: hai tập con khác nhau nếu **chỉ số** khác nhau. `[2,2,2]` có 7 tập con, không phải 1. **Sửa:** liệt kê theo chỉ số (mask hoặc DP từng phần tử).

### Lỗi 2 — Lấy `max(nums)` làm OR lớn nhất

In [22]:
def wrong_max(nums):
    t = max(nums); n = len(nums)
    return sum(1 for m in range(1, 1 << n) if reduce(or_, (nums[i] for i in range(n) if m >> i & 1)) == t)   # SAI

for nums, exp in examples:
    print(f"{nums}: sai {wrong_max(nums)} | đúng {exp}")

[3, 1]: sai 2 | đúng 2
[2, 2, 2]: sai 7 | đúng 7
[3, 2, 1, 5]: sai 2 | đúng 6


**Vì sao sai:** OR của nhiều số có thể **lớn hơn** số lớn nhất (`3 | 5 = 7 > 5`). **Sửa:** `target = OR` toàn mảng.

### Lỗi 3 — Cắt tỉa sai: trả `1` khi đạt target

In [23]:
def wrong_prune(nums):
    n = len(nums); t = reduce(or_, nums)
    def dfs(i, cur):
        if cur == t:
            return 1                                            # SAI: bỏ sót các tập lớn hơn
        if i == n:
            return 0
        return dfs(i + 1, cur | nums[i]) + dfs(i + 1, cur)
    return dfs(0, 0)

for nums, exp in examples:
    print(f"{nums}: sai {wrong_prune(nums)} | đúng {exp}")

[3, 1]: sai 1 | đúng 2
[2, 2, 2]: sai 3 | đúng 7
[3, 2, 1, 5]: sai 6 | đúng 6


**Vì sao sai:** khi `cur = target` ở bước `i`, vẫn còn `2^(n−i)` cách chọn phần còn lại (mỗi cách là một tập con **khác**). **Sửa:** `return 1 << (n − i)`, hoặc không cắt tỉa và chỉ đếm ở lá.

### Lỗi 4 — `range(1, n)` trong `combinations` (quên tập con đầy đủ)

In [24]:
def wrong_range(nums):
    n = len(nums); t = reduce(or_, nums)
    return sum(1 for r in range(1, n) for c in combinations(nums, r) if reduce(or_, c) == t)   # SAI: thiếu r = n

for nums, exp in examples:
    print(f"{nums}: sai {wrong_range(nums)} | đúng {exp}")

[3, 1]: sai 1 | đúng 2
[2, 2, 2]: sai 6 | đúng 7
[3, 2, 1, 5]: sai 5 | đúng 6


**Vì sao sai:** `range(1, n)` dừng ở `n − 1`, bỏ mất **chính mảng** — tập con luôn đạt target. Với `n = 1` còn trả `0`. **Sửa:** `range(1, n + 1)`. (`combinations` theo vị trí nên đếm đúng chỉ số.)

### Lỗi 5 — Nhầm `^` (XOR) với `|` (OR) khi gộp

In [25]:
def wrong_xor(nums):
    dp = {0: 1}
    for x in nums:
        nd = dict(dp)
        for v, c in dp.items(): nd[v ^ x] = nd.get(v ^ x, 0) + c      # SAI: phải là v | x
        dp = nd
    return dp.get(reduce(or_, nums), 0)

for nums, exp in examples:
    print(f"{nums}: sai {wrong_xor(nums)} | đúng {exp}")

[3, 1]: sai 1 | đúng 2
[2, 2, 2]: sai 4 | đúng 7
[3, 2, 1, 5]: sai 2 | đúng 6


**Vì sao sai:** `3 ^ 1 = 2` nhưng `3 | 1 = 3`; XOR **tắt** bit trùng nên `[3,1]` chỉ còn `[3]` có "giá trị" 3. Lỗi gõ phím kiểu này rất hay gặp trong bài bit. **Sửa:** dùng đúng `|`; test với phần tử có bit chung (như `[3,1]`).

### Lỗi 6 — Sửa `dp` trong lúc duyệt nó

In [26]:
def wrong_inplace2(nums):
    dp = defaultdict(int); dp[0] = 1
    for x in nums:
        for v in sorted(dp):                                    # SAI: đọc dp[v] SAU khi có thể đã bị cộng trong cùng bước
            dp[v | x] += dp[v]
    return dp[reduce(or_, nums)]

for nums, exp in examples:
    print(f"{nums}: sai {wrong_inplace2(nums)} | đúng {exp}")

[3, 1]: sai 2 | đúng 2
[2, 2, 2]: sai 10 | đúng 7
[3, 2, 1, 5]: sai 7 | đúng 6


**Vì sao sai:** vòng lặp đọc `dp[v]` **trực tiếp**: khóa `v` nhỏ đã cộng vào `w = v | x`, rồi tới lượt `w` thì giá trị **đã bị cập nhật** lại được cộng tiếp ⇒ phần tử `x` bị dùng nhiều lần trong cùng một bước (giống knapsack không giới hạn). Với `[2,2,2]`: bước thứ hai cho `dp[2] = 4` thay vì `3`. **Sửa:** cập nhật trên bản sao (`nd = dict(dp)`), hoặc chụp trước các cặp `list(dp.items())` rồi mới cộng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- `n ≤ 16`–`20` ⇒ liệt kê tập con bằng bitmask là hợp lệ.
- "Giá trị lớn nhất của OR/AND trên tập con" ⇒ OR lớn nhất là OR toàn bộ; AND lớn nhất là `max` (một phần tử), AND nhỏ nhất là AND toàn bộ.
- DP trên mask: `f[mask] = f[mask ^ lowbit] ⊕ a[idx(lowbit)]` để tính mọi tổng/OR/XOR tập con trong `O(2ⁿ)`.
- Đếm tập con theo giá trị tổng hợp ⇒ `dp[giá trị] = số tập con`, cập nhật trên **bản sao** (0/1 knapsack).
- Cắt tỉa khi đạt "trạng thái bão hòa": cộng `2^(còn lại)`.

## 15. Biến thể của bài

1. **Đếm tập con có OR = `k` bất kỳ** — vẫn `dp[k]` (Bài tập 4).
2. **78 Subsets / 90 Subsets II**: liệt kê tập con, có/không trùng giá trị.
3. **1863 Sum of All Subset XOR Totals**: tổng XOR mọi tập con — có công thức `OR(nums) · 2^(n−1)`.
4. **2275 Largest Combination With Bitwise AND Greater Than Zero**: đếm theo từng bit.
5. **`n` lớn (`10⁵`), giá trị nhỏ (`< 2²⁰`)**: đếm tập con có OR = target bằng **bao hàm–loại trừ trên bit** (SOS DP) — Bài tập 5.

### 15.1 Code biến thể: tổng XOR mọi tập con (1863) — công thức và kiểm chứng

In [27]:
def subset_xor_sum_brute(nums):
    n = len(nums); total = 0
    for m in range(1 << n):
        v = 0
        for i in range(n):
            if m >> i & 1: v ^= nums[i]
        total += v
    return total

def subset_xor_sum_formula(nums):
    return reduce(or_, nums) << (len(nums) - 1)        # mỗi bit có mặt trong OR xuất hiện ở đúng nửa số tập con

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 10))]
    assert subset_xor_sum_brute(nums) == subset_xor_sum_formula(nums)
print("1863: công thức OR · 2^(n−1) khớp vét cạn;", subset_xor_sum_formula([5, 1, 6]), "(LeetCode 1863 ví dụ: 28)")

1863: công thức OR · 2^(n−1) khớp vét cạn; 28 (LeetCode 1863 ví dụ: 28)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi mask | OR từng mask | `O(n·2ⁿ)` | `O(1)` | Kiểm chứng |
| 2. DFS cắt tỉa | Chọn/không chọn, `2^(n−i)` | `O(2ⁿ)` | `O(n)` | Ngắn, nhanh khi bão hòa sớm |
| 3. DP mask | `lowbit` | `O(2ⁿ)` | `O(2ⁿ)` | Cần OR của mọi mask |
| 4. DP theo giá trị | `dp[or]` | `O(n·K)` | `O(K)` | **Bản nộp**, tổng quát hóa tốt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1]`; (b) `[1, 2, 4]`; (c) `[5, 5]`; (d) `[1, 2, 3]`.

**Bài 2 (tính tay):** Trace DP theo giá trị cho `[6, 3, 1]` và cho biết đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `or_of_mask(nums, mask)`.

**Bài 4 (tự viết lại):** Viết `count_or_equals(nums, k)` — số tập con khác rỗng có OR đúng bằng `k` (0 nếu `k` không đạt được); đối chiếu vét cạn.

**Bài 5 (mini challenge):** Với `n` tới `10⁵` và `nums[i] < 2^B` (`B ≤ 12`), đếm số tập con khác rỗng có OR = `target` bằng bao hàm–loại trừ: gọi `g[m]` = số phần tử là tập con bit của `m`; số tập con khác rỗng có OR ⊆ `m` là `2^{g[m]} − 1`; đáp án `Σ_{m ⊆ target} (−1)^{|target| − |m|} (2^{g[m]} − 1)`. Tính `g` bằng SOS DP; trả kết quả modulo `10⁹ + 7`; đối chiếu DP theo giá trị trên dữ liệu nhỏ.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def or_of_mask(nums, mask):
    v = 0
    # TODO: với mỗi i, nếu bit i của mask bật thì v |= nums[i]
    return v

In [29]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def count_or_equals(nums, k):
    # TODO: DP theo giá trị như Cách 4, trả dp.get(k, 0) (nhớ trừ tập rỗng khi k == 0)
    pass

def count_max_or_sos(nums, B, mod=10 ** 9 + 7):
    # TODO: g[m] = số phần tử x với x ⊆ m (SOS DP); bao hàm–loại trừ trên các m ⊆ target
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: target = OR toàn mảng. Mức 2: đếm tập con đạt. Mức 3: (a) `1`; (b) `1` (phải chọn cả ba); (c) `3`; (d) `5` (`[3]`, `[1,2]`, `[1,3]`, `[2,3]`, `[1,2,3]` đều có OR 3).

**Bài 2.** Mức 1: `target = 6|3|1 = 7`. Mức 2: sau `6`: `{0:1, 6:1}`; sau `3`: `{0:1, 3:1, 6:1, 7:1}`. Mức 3: sau `1`: `{0:1, 1:1, 3:2, 6:1, 7:3}` ⇒ `3`.

**Bài 3.** Mức 1: duyệt `i`. Mức 2: `mask >> i & 1`. Mức 3: hoặc duyệt lowbit.

**Bài 4.** Mức 1: `dp` như Cách 4. Mức 2: `k` không phải tập con bit của OR toàn mảng ⇒ 0. Mức 3: `k = 0` ⇒ mọi phần tử ≥ 1 nên chỉ có tập rỗng ⇒ trả 0.

**Bài 5.** Mức 1: `g[m] = #{x : x | m == m}`; SOS: `for b: for m: if m >> b & 1: g[m] += g[m ^ (1 << b)]`. Mức 2: `F(m) = 2^{g[m]} − 1` = số tập con khác rỗng có OR ⊆ `m`. Mức 3: Möbius: `ans = Σ_{m ⊆ target} (−1)^{popcount(target ^ m)} F(m)`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, 2, 3
f = Solution().countMaxOrSubsets
b1 = [f(x) for x in ([1], [1, 2, 4], [5, 5], [1, 2, 3])]
print("Bài 1:", b1); assert b1 == [1, 1, 3, 5]
print("Bài 2:"); assert trace_dp([6, 3, 1]) == 3
def or_of_mask(nums, mask):
    v = 0
    for i in range(len(nums)):
        if mask >> i & 1:
            v |= nums[i]
    return v
assert [or_of_mask([3, 2, 1, 5], m) for m in (0b0001, 0b1001, 0b1111)] == [3, 7, 7]
print("Bài 3: OK")

Bài 1: [1, 1, 3, 5]
Bài 2:
ban đầu: {0: 1}
thêm 6: {0: 1, 6: 1}
thêm 3: {0: 1, 3: 1, 6: 1, 7: 1}
thêm 1: {0: 1, 1: 1, 3: 2, 6: 1, 7: 3}
target = 7 -> dp[target] = 3
Bài 3: OK


In [31]:
# Bài 4, 5
def count_or_equals(nums, k):
    if k == 0:
        return 0
    dp = {0: 1}
    for x in nums:
        nd = dict(dp)
        for v, c in dp.items(): nd[v | x] = nd.get(v | x, 0) + c
        dp = nd
    return dp.get(k, 0)

def count_max_or_sos(nums, B, mod=10 ** 9 + 7):
    size = 1 << B
    g = [0] * size
    for x in nums: g[x] += 1
    for b in range(B):
        for m in range(size):
            if m >> b & 1: g[m] += g[m ^ (1 << b)]
    target = reduce(or_, nums)
    pw = [1] * (len(nums) + 1)
    for i in range(1, len(nums) + 1): pw[i] = pw[i - 1] * 2 % mod
    ans, sub = 0, target
    while True:
        term = (pw[g[sub]] - 1) % mod
        ans = (ans - term) % mod if bin(target ^ sub).count("1") & 1 else (ans + term) % mod
        if sub == 0: break
        sub = (sub - 1) & target
    return ans

rng = random.Random(5)
for _ in range(300):
    nums = [rng.randint(1, 15) for _ in range(rng.randint(1, 10))]; k = rng.randint(0, 15)
    brute = sum(1 for m in range(1, 1 << len(nums)) if or_of_mask(nums, m) == k)
    assert count_or_equals(nums, k) == brute
    assert count_max_or_sos(nums, 4) == Solution().countMaxOrSubsets(nums) % (10 ** 9 + 7)
big = [rng.randint(1, 4095) for _ in range(10 ** 5)]
t = time.perf_counter(); r = count_max_or_sos(big, 12)
print(f"Bài 4, 5: OK | n = 10^5, B = 12: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")

Bài 4, 5: OK | n = 10^5, B = 12: 332004820 (14 ms)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (DP theo giá trị OR):**

```python
dp = {0: 1}
for x in nums:
    nd = dict(dp)                         # bản sao: không chọn x
    for v, c in dp.items():
        nd[v | x] = nd.get(v | x, 0) + c  # chọn x
    dp = nd
target = OR toàn mảng
return dp[target]
```

**Template (DFS cắt tỉa):** `dfs(i, cur)`: nếu `cur == target` trả `2^(n−i)`; nếu `i == n` trả 0; ngược lại chọn + không chọn.

**Checklist:**
- [ ] `target` là OR **toàn mảng** (không phải `max`)?
- [ ] Đếm theo **chỉ số** (không loại trùng giá trị)?
- [ ] Không đếm tập rỗng; có đếm tập đầy đủ?
- [ ] Cắt tỉa cộng `2^(n−i)`, không phải `1`?
- [ ] Cập nhật DP trên bản sao?